# Incubation Portfolio Optimizer
## A Policy-Aligned Pipeline for Evidence-Based Venture Selection

This notebook implements a reproducible two-stage pipeline for incubator portfolio design
under government-mandated multi-objective constraints:

**Stage 1 — Predictive Modeling**
- Ridge-penalized logistic regression per outcome, penalty selected by LOOCV (log-loss)
- Out-of-sample performance: AUC, Log-Loss, Brier score via LOOCV
- Permutation tests for significance of predictive signal
- Calibration assessment: reliability curves, Brier decomposition, isotonic post-calibration
- Average Marginal Effects (AMEs) with bootstrap 95% CIs

**Stage 2 — Portfolio Optimization**
- Per-venture objective coefficients from AMEs
- Multi-Objective Binary Integer Program (MO-BIP) solved with PuLP/CBC
- ε-constraint method to trace Pareto frontiers under capacity and composition constraints
- Monte Carlo uncertainty propagation (AME + baseline sampling)
- Evidence-weighted objectives (optional: scale by permutation p-value confidence)

**Outcomes** (binary, mapped to government KPI categories):

| Variable | Policy KPI |
|---|---|
| `ip` | Intellectual property / technology transfer |
| `hub` | Regional anchoring / science park affiliation |
| `rev_high` | Revenue / commercialization impact |
| `team_growth` | Employment generation |
| `stage_growth` | Technological stage advancement |

**Predictors**:

| Variable | Type | Description |
|---|---|---|
| `tech_digital` | binary | Tech domain: 1=digital/platform/Industry-4.0, 0=bio/agri |
| `founders` | integer | Number of founding partners |
| `incub_years` | numeric | Incubation duration (years) |
| `stage_in` | ordinal (1–4) | Technology stage at entry: 1=Ideation → 4=Scale-up |
| `team_size_in` | binary | Team size bracket at entry (0=<5, 1=5–10 employees) |
| `stage_adv` | binary | Advanced stage at entry: `stage_in` > 2 |

**Usage**: Set `DATA_PATH` in *Section 1* to your cleaned dataset (`data/venture_cohort.csv`).
Adjust `OUTCOME_SPECS` to match your covariates.
All random operations are seeded for full reproducibility.

---
*Reference*: [Author(s) omitted for review]. Designing Incubation Portfolios Aligned
with Public Policy Objectives. *Technovation* (under review).


## Section 1 — Setup & Configuration

In [ ]:
# ── Standard library & scientific stack ──────────────────────────────────────import warnings, jsonwarnings.simplefilter("ignore", UserWarning)from pathlib import Pathimport numpy as npimport pandas as pdfrom pandas.api.types import CategoricalDtypefrom sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import LeaveOneOutfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import roc_auc_score, log_loss, brier_score_lossfrom sklearn.calibration import calibration_curve, CalibratedClassifierCVfrom scipy.special import expitimport matplotlib.pyplot as pltimport matplotlib.gridspec as gridspectry:    import pulpexcept ImportError:    raise ImportError("PuLP is required: pip install pulp")

In [ ]:
# ── File paths (update DATA_PATH to your cleaned CSV) ────────────────────────
DATA_PATH      = Path("data/venture_cohort.csv")        # output of data_preparation step
DICT_PATH      = Path("data/venture_cohort_dictionary.json")
OUTPUTS_DIR    = Path("outputs")
OUTPUTS_DIR.mkdir(exist_ok=True)
# ── Reproducibility seeds ─────────────────────────────────────────────────────
SEED_PERM  = 42     # permutation tests
SEED_BOOT  = 123    # pairs bootstrap
SEED_MC    = 42     # Monte Carlo
# ── Hyperparameters ───────────────────────────────────────────────────────────
C_GRID    = np.logspace(-3, 3, 25)    # ridge penalty search grid (C = 1/lambda)
B_PERM    = 2_000                      # permutation draws
R_BOOT    = 2_000                      # bootstrap resamples
MC_DRAWS  = 2_000                      # Monte Carlo draws
CAPACITY  = 8                          # incubator slot capacity (K)
ALPHAS    = [0.4, 0.6, 0.8, 1.0]      # Expand-vs-Admit gain scaling
EPS_VEC   = np.linspace(0.05, 0.95, 19)  # epsilon-constraint grid
print("Configuration loaded.")
print(f"  Data : {DATA_PATH}")
print(f"  B_PERM={B_PERM}, R_BOOT={R_BOOT}, MC_DRAWS={MC_DRAWS}, K={CAPACITY}")


In [ ]:
# ── Outcome specification: which predictors enter each model ─────────────────
#    bin_cols  : binary predictors (0/1), kept as-is
#    num_cols  : continuous/ordinal predictors, standardized (mean=0, sd=1)
#    cat_cols  : categorical predictors, one-hot encoded (drop_first=True)
#
#    Variable names match the columns in venture_cohort.csv.
#    Adapt this dict to your own dataset's column names.
OUTCOME_SPECS = {
    "ip":           {"bin_cols": ["tech_digital"],             "num_cols": ["founders"],     "cat_cols": []},
    "hub":          {"bin_cols": [],                           "num_cols": ["incub_years"],  "cat_cols": ["stage_in"]},
    "rev_high":     {"bin_cols": ["team_size_in"],             "num_cols": ["incub_years"],  "cat_cols": []},
    "team_growth":  {"bin_cols": ["tech_digital", "stage_adv"],"num_cols": [],               "cat_cols": []},
    "stage_growth": {"bin_cols": ["stage_adv", "team_size_in"],"num_cols": [],               "cat_cols": []},
}
PRIMARY   = "rev_high"
ALL_OBJS  = list(OUTCOME_SPECS.keys())
SECONDARY = [o for o in ALL_OBJS if o != PRIMARY]
print("Outcome specs:", list(OUTCOME_SPECS.keys()))


## Section 2 — Data Preparation

Load the cleaned dataset produced by `data_preparation.py` (or the companion
data-preparation notebook). The expected schema is documented in
`data/venture_cohort_dictionary.json`.

**Binary outcomes** (0/1):
- `ip` — intellectual property registered (patent/software) during or after graduation
- `hub` — post-graduation affiliation with hub/park/innovation space
- `rev_high` — high-revenue bracket (≥ R$5M–R$10M)
- `team_growth` — team headcount bracket grew after graduation
- `stage_growth` — technology maturity stage advanced after graduation

**Predictors**:
- `tech_digital` (0/1): digital/platform/Industry-4.0 tech domain
- `founders` (int): number of founding partners
- `incub_years` (float): incubation duration in years
- `stage_in` (1–4): technology maturity stage at incubation entry
- `team_size_in` (0/1): entry team size bracket (0=<5, 1=5–10)
- `stage_adv` (0/1): advanced entry stage (stage_in > 2)


In [ ]:
df = pd.read_csv(DATA_PATH)print(f"Dataset: {len(df)} ventures × {df.shape[1]} columns")print(f"\nOutcome prevalences:")for col in ALL_OBJS:    if col in df.columns:        p = pd.to_numeric(df[col], errors="coerce").mean()        print(f"  {col:12s}: {p:.3f}  ({int(round(p*len(df)))}/{len(df)})")df.head()

## Section 3 — Helper FunctionsAll methodological steps are encapsulated here. These functions are calledonce per outcome in Section 4 and are fully reusable for other datasets.

In [ ]:
def build_design_matrix(df, spec):    """    Build standardized design matrix X and response vector y for a single outcome.    Parameters    ----------    df   : pd.DataFrame  — cleaned dataset    spec : dict          — {"target": str, "bin_cols": [...], "num_cols": [...], "cat_cols": [...]}    Returns    -------    X          : np.ndarray (n, p)    y          : np.ndarray (n,)    scaler     : fitted StandardScaler (for num_cols)    num_used   : list of continuous column names in X    bin_used   : list of binary column names in X    cat_dummies: list of dummy column names from cat_cols    """    target   = spec["target"]    bin_cols = spec.get("bin_cols", [])    num_cols = spec.get("num_cols", [])    cat_cols = spec.get("cat_cols", [])    parts, num_used, bin_used, cat_dummies = [], [], [], []    # Continuous / ordinal (standardized)    scaler = StandardScaler()    if num_cols:        Xn = df[num_cols].apply(pd.to_numeric, errors="coerce")        Xn_scaled = scaler.fit_transform(Xn)        parts.append(Xn_scaled)        num_used = num_cols[:]    # Binary predictors (kept as 0/1)    if bin_cols:        Xb = df[bin_cols].apply(pd.to_numeric, errors="coerce").values        parts.append(Xb)        bin_used = bin_cols[:]    # Categorical (one-hot, drop first to avoid collinearity)    for col in cat_cols:        dummies = pd.get_dummies(df[col].astype(str), drop_first=True,                                 dtype=float, prefix=col)        parts.append(dummies.values)        cat_dummies += list(dummies.columns)    X = np.hstack(parts).astype(float)    y = pd.to_numeric(df[target], errors="coerce").values.astype(float)    # Drop rows with any NaN    mask = np.isfinite(X).all(axis=1) & np.isfinite(y)    return X[mask], y[mask], scaler, num_used, bin_used, cat_dummies

In [ ]:
def fit_loocv_ridge(X, y, C_grid):    """    Select ridge penalty C by leave-one-out cross-validation (log-loss criterion).    Returns best_C (float) and grid_df (DataFrame with C vs loocv_logloss).    """    loo = LeaveOneOut()    grid = []    for C in C_grid:        clf = LogisticRegression(penalty="l2", C=C, solver="lbfgs", max_iter=5000)        y_true, y_pred = [], []        try:            for tr, te in loo.split(X):                clf.fit(X[tr], y[tr])                y_pred.append(clf.predict_proba(X[te])[:, 1][0])                y_true.append(y[te][0])            ll = log_loss(y_true, y_pred, labels=[0, 1])        except Exception:            ll = np.inf        grid.append((C, ll))    grid_df = pd.DataFrame(grid, columns=["C", "loocv_logloss"])    best_C = float(grid_df.loc[grid_df["loocv_logloss"].idxmin(), "C"])    return best_C, grid_df

In [ ]:
def compute_loocv_metrics_and_permtest(X, y, best_C, B=2_000, seed=42):    """    Compute LOOCV metrics (AUC, LogLoss, Brier) and permutation p-values.    Also assesses probability calibration via reliability curves and    Brier decomposition; applies isotonic post-calibration when miscalibrated.    Parameters    ----------    X      : np.ndarray    y      : np.ndarray    best_C : float — ridge penalty    B      : int   — number of permutation draws    seed   : int    Returns    -------    metrics : dict with observed and null-distribution results    calib   : dict with calibration diagnostics    """    loo = LeaveOneOut()    rng = np.random.default_rng(seed)    def _loocv_probs(X_, y_, C):        clf = LogisticRegression(penalty="l2", C=C, solver="lbfgs", max_iter=5000)        yt, yp = [], []        for tr, te in loo.split(X_):            clf.fit(X_[tr], y_[tr])            yp.append(clf.predict_proba(X_[te])[:, 1][0])            yt.append(y_[te][0])        return np.array(yt), np.array(yp)    y_true, y_prob = _loocv_probs(X, y, best_C)    # ── Observed LOOCV metrics ────────────────────────────────────────────────    obs_auc = roc_auc_score(y_true, y_prob)    obs_ll  = log_loss(y_true, y_prob, labels=[0, 1])    obs_br  = brier_score_loss(y_true, y_prob)    # ── Permutation nulls ─────────────────────────────────────────────────────    null_auc, null_ll, null_br = [], [], []    for _ in range(B):        yp_ = rng.permutation(y)        try:            _, yh = _loocv_probs(X, yp_, best_C)            null_auc.append(roc_auc_score(yp_, yh))            null_ll.append(log_loss(yp_, yh, labels=[0, 1]))            null_br.append(brier_score_loss(yp_, yh))        except Exception:            pass    def pval_hi(obs, null):  # higher is better (AUC)        return (1 + np.sum(np.array(null) >= obs)) / (len(null) + 1)    def pval_lo(obs, null):  # lower is better (LL, Brier)        return (1 + np.sum(np.array(null) <= obs)) / (len(null) + 1)    metrics = {        "AUC":   obs_auc, "p_AUC":   pval_hi(obs_auc, null_auc),        "LogLoss": obs_ll, "p_LL":   pval_lo(obs_ll,  null_ll),        "Brier":  obs_br,  "p_Brier": pval_lo(obs_br,  null_br),        "y_true": y_true, "y_prob": y_prob,    }    # ── Calibration assessment ────────────────────────────────────────────────    # Brier decomposition: Brier = Reliability - Resolution + Uncertainty    p_bar = y_true.mean()    uncertainty = p_bar * (1 - p_bar)    n_bins = max(5, len(y_true) // 5)    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy="quantile")    # Reliability (calibration error): E[(p_hat - p_obs)^2] weighted by bin counts    counts = np.histogram(y_prob, bins=n_bins)[0]    reliability = np.average((mean_pred - frac_pos) ** 2, weights=counts + 1e-9)    resolution  = np.average((frac_pos - p_bar) ** 2,  weights=counts + 1e-9)    # Isotonic post-calibration (applied when reliability > 0.02)    isotonic_probs = None    is_miscalibrated = reliability > 0.02    if is_miscalibrated:        base_clf = LogisticRegression(penalty="l2", C=best_C, solver="lbfgs", max_iter=5000)        cal_clf  = CalibratedClassifierCV(base_clf, method="isotonic", cv=LeaveOneOut())        cal_clf.fit(X, y)        isotonic_probs = cal_clf.predict_proba(X)[:, 1]    calib = {        "reliability": reliability, "resolution": resolution,        "uncertainty": uncertainty, "is_miscalibrated": is_miscalibrated,        "frac_pos": frac_pos, "mean_pred": mean_pred,        "isotonic_probs": isotonic_probs,    }    return metrics, calib

In [ ]:
def compute_ames_with_bootstrap(clf, X, y, num_cols_used, bin_cols_used, scaler,                               cat_dummies=None, R=2_000, seed=123):    """    Compute Average Marginal Effects (AMEs) with pairs-bootstrap 95% CIs.    Continuous/ordinal predictors: delta-method AME in probability units per 1 SD.    Binary predictors: average discrete change P(Y=1|x=1) - P(Y=1|x=0).    Returns ame_df with columns: [var, type, AME, CI_2.5, CI_97.5].    """    cat_dummies = cat_dummies or []    beta = clf.coef_.ravel()    n_num, n_bin = len(num_cols_used), len(bin_cols_used)    def _ames_once(clf_, X_, scaler_=None):        mu  = clf_.predict_proba(X_)[:, 1]        b   = clf_.coef_.ravel()        rows = []        # Continuous        for j, col in enumerate(num_cols_used):            sd = scaler_.scale_[j] if (scaler_ is not None and scaler_.scale_ is not None) else 1.0            dmu = mu * (1 - mu) * (b[j] / sd)            rows.append((col, "continuous", float(np.mean(dmu))))        # Binary        offset = n_num        for k, col in enumerate(bin_cols_used):            j = offset + k            X0, X1 = X_.copy(), X_.copy()            X0[:, j], X1[:, j] = 0.0, 1.0            mu0 = clf_.predict_proba(X0)[:, 1]            mu1 = clf_.predict_proba(X1)[:, 1]            rows.append((col, "binary", float(np.mean(mu1 - mu0))))        return rows    ame_rows = _ames_once(clf, X, scaler)    ame_df   = pd.DataFrame(ame_rows, columns=["var", "type", "AME"])    # Bootstrap    rng  = np.random.default_rng(seed)    boot = {r[0]: [] for r in ame_rows}    for _ in range(R):        idx = rng.integers(0, len(X), size=len(X))        Xb, yb = X[idx], y[idx]        sc_b = StandardScaler().fit(Xb[:, :n_num]) if n_num > 0 else None        try:            clf_b = LogisticRegression(penalty="l2", C=clf.C, solver="lbfgs", max_iter=5000)            clf_b.fit(Xb, yb)            for var, _, val in _ames_once(clf_b, Xb, sc_b):                boot[var].append(val)        except Exception:            pass    ci_rows = []    for var, draws in boot.items():        if draws:            ci_rows.append({"var": var,                            "CI_2.5":  float(np.percentile(draws, 2.5)),                            "CI_97.5": float(np.percentile(draws, 97.5))})    ci_df  = pd.DataFrame(ci_rows).set_index("var")    ame_df = ame_df.set_index("var").join(ci_df, how="left").reset_index()    return ame_df

In [ ]:
def run_outcome_pipeline(target_col, df, C_grid=C_GRID, B_perm=B_PERM,                         R_boot=R_BOOT, seed_perm=SEED_PERM, seed_boot=SEED_BOOT,                         save_prefix=None):    """    Full modeling pipeline for one binary outcome:    1. Build design matrix from OUTCOME_SPECS[target_col]    2. Select ridge penalty by LOOCV    3. Fit final model    4. Compute LOOCV metrics + permutation test + calibration    5. Compute AMEs with bootstrap CIs    6. Save CSVs and return results dict    Parameters    ----------    target_col  : str   — key in OUTCOME_SPECS    df          : DataFrame    save_prefix : str or None — prefix for CSV outputs (None = no saving)    Returns    -------    results : dict with keys: metrics, calib, ame_df, clf, X, y, scaler, spec    """    spec = dict(OUTCOME_SPECS[target_col])    spec["target"] = target_col    X, y, scaler, num_used, bin_used, cat_dummies = build_design_matrix(df, spec)    print(f"\n{'─'*55}")    print(f"Outcome: {target_col.upper()}  |  n={len(y)}, p0={y.mean():.3f}")    print(f"  Predictors — num:{num_used}  bin:{bin_used}  cat:{cat_dummies}")    # 1) LOOCV penalty selection    best_C, grid_df = fit_loocv_ridge(X, y, C_grid)    print(f"  Best C={best_C:.4f} (LOOCV log-loss={grid_df['loocv_logloss'].min():.4f})")    # 2) Final model    clf = LogisticRegression(penalty="l2", C=best_C, solver="lbfgs", max_iter=5000)    clf.fit(X, y)    # 3) LOOCV metrics + permutation test + calibration    metrics, calib = compute_loocv_metrics_and_permtest(X, y, best_C, B=B_perm, seed=seed_perm)    print(f"  AUC={metrics['AUC']:.4f} (p={metrics['p_AUC']:.4f}) | "          f"LL={metrics['LogLoss']:.4f} (p={metrics['p_LL']:.4f}) | "          f"Brier={metrics['Brier']:.4f} (p={metrics['p_Brier']:.4f})")    if calib["is_miscalibrated"]:        print(f"  ⚠ Miscalibration detected (reliability={calib['reliability']:.4f}); "              f"isotonic post-calibration applied.")    # 4) AMEs with bootstrap    ame_df = compute_ames_with_bootstrap(clf, X, y, num_used, bin_used, scaler,                                         cat_dummies=cat_dummies, R=R_boot, seed=seed_boot)    # 5) Save    if save_prefix:        pfx = OUTPUTS_DIR / save_prefix        ame_df.to_csv(f"{pfx}_ames.csv", index=False)        pd.DataFrame([{k: v for k, v in metrics.items() if k not in ("y_true","y_prob")}                     ]).to_csv(f"{pfx}_loocv_perm.csv", index=False)    return dict(metrics=metrics, calib=calib, ame_df=ame_df,                clf=clf, X=X, y=y, scaler=scaler,                num_used=num_used, bin_used=bin_used, spec=spec)print("Helper functions defined.")

## Section 4 — Stage 1: Predictive ModelingRun the full pipeline for each outcome. Results are stored in `RESULTS` dict.

In [ ]:
RESULTS = {}for outcome in ALL_OBJS:    RESULTS[outcome] = run_outcome_pipeline(        target_col  = outcome,        df          = df,        save_prefix = outcome,    )print("\nAll outcomes processed.")

In [ ]:
# ── Table: LOOCV metrics and permutation p-values (Table 2 in paper) ─────────rows = []for outcome, res in RESULTS.items():    m = res["metrics"]    rows.append({"Outcome": outcome,                 "AUC": m["AUC"],  "p_AUC": m["p_AUC"],                 "LogLoss": m["LogLoss"], "p_LL": m["p_LL"],                 "Brier": m["Brier"],   "p_Brier": m["p_Brier"]})pred_table = pd.DataFrame(rows).set_index("Outcome").round(4)pred_table.to_csv(OUTPUTS_DIR / "table_loocv_metrics.csv")print("LOOCV metrics table:")print(pred_table)

In [ ]:
# ── Table: AMEs with bootstrap CIs (Table 3 in paper) ────────────────────────ame_tables = {}for outcome, res in RESULTS.items():    ame_df = res["ame_df"].copy()    ame_df["Outcome"] = outcome    ame_tables[outcome] = ame_dfall_ames = pd.concat(ame_tables.values(), ignore_index=True)all_ames.to_csv(OUTPUTS_DIR / "table_ames.csv", index=False)print("AME table (all outcomes):")print(all_ames.to_string(index=False))

In [ ]:
# ── Calibration reliability curves ────────────────────────────────────────────fig, axes = plt.subplots(1, len(ALL_OBJS), figsize=(4*len(ALL_OBJS), 4), sharey=True)for ax, outcome in zip(axes, ALL_OBJS):    calib = RESULTS[outcome]["calib"]    ax.plot(calib["mean_pred"], calib["frac_pos"], "s-", label="LOOCV probs")    if calib["isotonic_probs"] is not None:        y_true = RESULTS[outcome]["metrics"]["y_true"]        frac_c, mean_c = calibration_curve(y_true, calib["isotonic_probs"],                                           n_bins=5, strategy="quantile")        ax.plot(mean_c, frac_c, "^--", label="Isotonic")    ax.plot([0,1],[0,1],"k--", alpha=0.4, label="Perfect")    ax.set_title(outcome); ax.set_xlabel("Mean predicted"); ax.legend(fontsize=7)axes[0].set_ylabel("Fraction positives")fig.suptitle("Reliability (Calibration) Curves — LOOCV predictions", y=1.01)plt.tight_layout()plt.savefig(OUTPUTS_DIR / "fig_calibration_curves.png", dpi=150, bbox_inches="tight")plt.show()print("Calibration curves saved.")

## Section 5 — Per-Venture Objective CoefficientsFor each venture *i* and outcome *j*:$$g_{ij} = p_{0j} + \sum_k \text{AME}_{jk} \cdot x_{ik}$$where $p_{0j}$ is the sample prevalence and AME medians are used as point estimates.These coefficients serve as the objective weights $u_{ij}$ in the MILP.

In [ ]:
def build_objective_coeffs(df, ame_df, target_col, scaler=None, num_used=None, bin_used=None):    """    Compute per-venture objective coefficient g_i = p0 + sum_k AME_k * x_ik.    Returns a Series indexed by df.index with g_i values.    """    y_vec = pd.to_numeric(df[target_col], errors="coerce")    p0    = float(y_vec.mean())    ame_map = dict(zip(ame_df["var"], ame_df["AME"]))    g = np.full(len(df), p0, dtype=float)    for var, ame_val in ame_map.items():        if var in df.columns:            x_col = pd.to_numeric(df[var], errors="coerce").fillna(0).values            g += ame_val * x_col    return pd.Series(g, index=df.index, name=target_col)df_coeffs = {"startup_id": np.arange(1, len(df)+1)}for outcome, res in RESULTS.items():    g = build_objective_coeffs(df, res["ame_df"], outcome,                               scaler=res["scaler"], num_used=res["num_used"],                               bin_used=res["bin_used"])    df_coeffs[outcome] = g.valuesCOEFFS = pd.DataFrame(df_coeffs)# Min-max normalize to [0,1] per objectiveNORM = COEFFS.copy()for col in ALL_OBJS:    lo, hi = COEFFS[col].min(), COEFFS[col].max()    NORM[col + "_01"] = (COEFFS[col] - lo) / (hi - lo) if hi > lo else 0.0NORM.to_csv(OUTPUTS_DIR / "objectives_normalized.csv", index=False)print("Per-venture coefficients computed and normalized.")print(NORM[[c + "_01" for c in ALL_OBJS]].describe().round(3))

## Section 6 — Stage 2: ε-Constraint Portfolio OptimizationSolves the capacity-constrained MO-BIP:$$\max_{x \in \{0,1\}^n} \sum_i u_{i,\text{primary}} x_i$$$$\text{s.t.} \quad \sum_i x_i = K, \quad \sum_i u_{ij} x_i \geq \varepsilon_j \; \forall j \neq \text{primary}$$**Evidence-weighted variant** (optional): scale each $u_{ij}$ by a confidencefactor derived from permutation p-values before optimization:$$\tilde{u}_{ij} = u_{ij} \cdot w_j, \quad w_j = 1 - p_j^{\text{perm}}$$Set `USE_EVIDENCE_WEIGHTS=True` to activate.

In [ ]:
# ── Evidence-weighting (optional) ─────────────────────────────────────────────USE_EVIDENCE_WEIGHTS = False   # set True for the evidence-weighted variantobj_cols_norm = [c + "_01" for c in ALL_OBJS]if USE_EVIDENCE_WEIGHTS:    # Weight each objective by (1 - permutation p-value for AUC)    ew = {}    for col, outcome in zip(obj_cols_norm, ALL_OBJS):        p_auc = RESULTS[outcome]["metrics"]["p_AUC"]        ew[col] = max(0.0, 1.0 - p_auc)    print("Evidence weights:", {k: round(v, 3) for k, v in ew.items()})else:    ew = {c: 1.0 for c in obj_cols_norm}    print("Evidence weights: uniform (1.0 per objective)")

In [ ]:
def solve_eps_constraint(DF_norm, primary_col, secondary_cols,                          K, eps_grid, obj_weights=None, verbose=False):    """    Sweep the ε-constraint MILP across a grid of secondary floors.    Parameters    ----------    DF_norm        : DataFrame with startup_id and normalized objective columns    primary_col    : str — name of primary objective column    secondary_cols : list[str] — secondary objectives    K              : int — slot capacity    eps_grid       : 1-D array of ε values (applied uniformly to all secondaries)    obj_weights    : dict[col -> float] — optional evidence weights    verbose        : bool    Returns    -------    frontier : DataFrame with columns [eps, status, gap, obj_value, selected, *secondary_sums]    """    if obj_weights is None:        obj_weights = {c: 1.0 for c in [primary_col] + secondary_cols}    ids = DF_norm["startup_id"].astype(int).tolist()    rows = []    for eps in eps_grid:        prob = pulp.LpProblem("eps_constraint", pulp.LpMaximize)        x    = {i: pulp.LpVariable(f"x_{i}", cat="Binary") for i in ids}        # Objective        prob += pulp.lpSum(            obj_weights.get(primary_col, 1.0) *            float(DF_norm.loc[DF_norm["startup_id"]==i, primary_col].values[0]) * x[i]            for i in ids)        # Capacity        prob += (pulp.lpSum(x[i] for i in ids) == K), "capacity"        # ε-floors on secondaries        for sec in secondary_cols:            prob += (pulp.lpSum(                obj_weights.get(sec, 1.0) *                float(DF_norm.loc[DF_norm["startup_id"]==i, sec].values[0]) * x[i]                for i in ids) >= eps * K), f"floor_{sec}"        prob.solve(pulp.PULP_CBC_CMD(msg=int(verbose)))        status   = pulp.LpStatus[prob.status]        obj_val  = pulp.value(prob.objective) if status == "Optimal" else np.nan        selected = [i for i in ids if pulp.value(x[i]) and pulp.value(x[i]) > 0.5] \                   if status == "Optimal" else []        sec_sums = {}        for sec in secondary_cols:            if status == "Optimal":                sec_sums[sec] = sum(                    float(DF_norm.loc[DF_norm["startup_id"]==i, sec].values[0])                    for i in selected)            else:                sec_sums[sec] = np.nan        rows.append({"eps": eps, "status": status, "obj_value": obj_val,                     "selected": selected, **sec_sums})    return pd.DataFrame(rows)print("MILP solver function defined.")

In [ ]:
# ── Run ε-constraint sweep ────────────────────────────────────────────────────PRIMARY_COL   = PRIMARY + "_01"SECONDARY_COLS= [c + "_01" for c in SECONDARY]frontier = solve_eps_constraint(    DF_norm        = NORM,    primary_col    = PRIMARY_COL,    secondary_cols = SECONDARY_COLS,    K              = CAPACITY,    eps_grid       = EPS_VEC,    obj_weights    = ew,)feasible  = frontier[frontier["status"]=="Optimal"]n_total   = len(frontier)n_feasible= len(feasible)print(f"Instances: {n_total} | Feasible: {n_feasible} ({n_feasible/n_total:.1%})")print(f"Optimality gap: always 0.00 for feasible instances (CBC exact solver)")frontier.to_csv(OUTPUTS_DIR / "frontier.csv", index=False)frontier.head(8)

In [ ]:
# ── Pareto front scatter ─────────────────────────────────────────────────────if len(feasible) > 0:    sec_mean = feasible[SECONDARY_COLS].mean(axis=1)    plt.figure(figsize=(6,4))    plt.scatter(sec_mean, feasible["obj_value"], alpha=0.7, edgecolors="k", s=60)    plt.xlabel("Mean of normalized secondary objectives")    plt.ylabel(f"Primary objective ({PRIMARY})")    plt.title("Pareto Front — ε-constraint sweep")    plt.tight_layout()    plt.savefig(OUTPUTS_DIR / "front_scatter.png", dpi=150, bbox_inches="tight")    plt.show()

## Section 7 — Stage 3: Monte Carlo Uncertainty PropagationPropagates uncertainty in AMEs (sampled from Normal distributions implied bybootstrap CIs) and baseline rates (Beta priors) across `MC_DRAWS` draws.For each draw, the ε-constraint problem is solved for a policy scenario comparingtwo candidate options (*Expand* an incumbent vs. *Admit* a waitlist entrant)across a grid of α (Expand gain scaling) and ε (secondary floor) values.**Outputs**: selection probabilities, feasibility surfaces, Pareto front bands,switching thresholds, and robust rankings.

In [ ]:
# ── AME uncertainty sampling ──────────────────────────────────────────────────def sample_ames(RESULTS, rng):    """    Sample one draw of AMEs per outcome from Normal(median, se_approx).    se_approx = (CI_97.5 - CI_2.5) / (2 * 1.96)    """    sampled = {}    for outcome, res in RESULTS.items():        ame_df = res["ame_df"].copy()        vals = []        for _, row in ame_df.iterrows():            lo, hi = row.get("CI_2.5", row["AME"]), row.get("CI_97.5", row["AME"])            se = (hi - lo) / (2 * 1.96) if (hi - lo) > 0 else 1e-6            vals.append(rng.normal(loc=row["AME"], scale=se))        sampled[outcome] = pd.DataFrame({"var": ame_df["var"].values, "AME": vals})    return sampled# ── Baseline rate sampling ─────────────────────────────────────────────────────def sample_baselines(df, ALL_OBJS, rng):    """Sample baseline prevalences p0 from Beta(a, b) priors."""    p0s = {}    for col in ALL_OBJS:        y_  = pd.to_numeric(df[col], errors="coerce").dropna()        a, b_ = float(y_.sum()) + 0.5, float((1-y_).sum()) + 0.5        p0s[col] = float(rng.beta(a, b_))    return p0sprint("Monte Carlo sampling functions defined.")

In [ ]:
# ── Define candidate options (Expand vs Admit) ───────────────────────────────# CUSTOMIZE: define the venture profiles for your policy scenario.# Each entry is a dict mapping predictor variables to their values for that venture.# Incumbent ventures are already incubated; waitlisted ventures are candidates.# The "alpha" parameter scales the effective gain credited to the Expand option# (alpha=1.0 means expansion benefits equal admission benefits).INCUMBENTS  = ["I1", "I2", "I7"]          # labels of incumbent venturesWAITLISTED  = ["W2", "W3"]                # labels of waitlisted ventures# ── Load observed cohort for baseline probability estimation ──────────────────obs_df = df.copy()obs_df["startup_id"] = np.arange(1, len(obs_df)+1)print(f"Policy scenario: Expand ({INCUMBENTS}) vs Admit ({WAITLISTED})")print(f"MC draws: {MC_DRAWS}, alpha grid: {ALPHAS}, eps grid length: {len(EPS_VEC)}")

In [ ]:
# ── Monte Carlo main loop ─────────────────────────────────────────────────────rng_mc  = np.random.default_rng(SEED_MC)log_rows = []for draw in range(MC_DRAWS):    sampled_ames = sample_ames(RESULTS, rng_mc)    p0s          = sample_baselines(obs_df, ALL_OBJS, rng_mc)    # Build per-startup coefficients from this draw's AMEs and baselines    g_draw = {"startup_id": obs_df["startup_id"].tolist()}    for outcome in ALL_OBJS:        ame_df_draw = sampled_ames[outcome]        g = build_objective_coeffs(obs_df, ame_df_draw, outcome)        g = g - g.mean() + p0s[outcome]          # re-center to sampled baseline        g_draw[outcome] = np.clip(g.values, 0, 1)    DF_draw = pd.DataFrame(g_draw)    # Normalize    DF_draw_norm = DF_draw.copy()    for col in ALL_OBJS:        lo, hi = DF_draw[col].min(), DF_draw[col].max()        DF_draw_norm[col+"_01"] = (DF_draw[col]-lo)/(hi-lo) if hi>lo else 0.0    for alpha in ALPHAS:        # Scale Expand candidates by alpha        DF_scenario = DF_draw_norm.copy()        for col in [c+"_01" for c in ALL_OBJS]:            expand_mask = DF_scenario["startup_id"].isin(                [i for i, lbl in enumerate(INCUMBENTS, 1) if True])            DF_scenario.loc[expand_mask, col] *= alpha        frontier_draw = solve_eps_constraint(            DF_norm=DF_scenario, primary_col=PRIMARY+"_01",            secondary_cols=[c+"_01" for c in SECONDARY],            K=CAPACITY, eps_grid=EPS_VEC, verbose=False)        for _, row in frontier_draw.iterrows():            log_rows.append({"draw": draw, "alpha": alpha, "eps": row["eps"],                             "status": row["status"], "obj_value": row.get("obj_value"),                             "selected": str(row.get("selected",""))})    if (draw+1) % 200 == 0:        print(f"  MC draw {draw+1}/{MC_DRAWS}")log_df = pd.DataFrame(log_rows)log_df.to_csv(OUTPUTS_DIR / "mc_selection_log.csv", index=False)print(f"Monte Carlo complete. {len(log_df)} rows logged.")

In [ ]:
# ── MC Summary: P(Feasible) and P(Expand) heatmaps ───────────────────────────agg = (log_df.groupby(["alpha","eps"])       .apply(lambda g: pd.Series({           "P_feasible": (g["status"]=="Optimal").mean(),           "P_expand":   (g["status"]=="Optimal").mean(),   # refine with your labeling           "rev_high":    g["obj_value"].mean(skipna=True),       })).reset_index())fig, axes = plt.subplots(1, 2, figsize=(12, 4))for ax, col, title in zip(axes,                           ["P_feasible", "rev_high"],                           ["P(Feasible)", "Mean primary obj (RevHigh)"]):    pivot = agg.pivot(index="alpha", columns="eps", values=col)    im = ax.imshow(pivot.values, aspect="auto", origin="lower",                   extent=[EPS_VEC.min(), EPS_VEC.max(),                           min(ALPHAS)-.1, max(ALPHAS)+.1])    plt.colorbar(im, ax=ax)    ax.set_xlabel("ε"); ax.set_ylabel("α"); ax.set_title(title)plt.suptitle("Monte Carlo — Policy Scenario Summary")plt.tight_layout()plt.savefig(OUTPUTS_DIR / "mc_heat_expand.png", dpi=150, bbox_inches="tight")plt.show()print("MC heatmaps saved.")

In [ ]:
# ── MC Summary: Switching thresholds ─────────────────────────────────────────threshold_rows = []for alpha, g in log_df[log_df["status"]=="Optimal"].groupby("alpha"):    switches = []    for draw_id, dg in g.groupby("draw"):        dg = dg.sort_values("eps")        # Detect first ε where dominant option switches (simplified)        switches.append(float(dg["eps"].min()))  # placeholder: customize per scenario    threshold_rows.append({        "alpha": alpha,        "eps_switch_median": np.median(switches),        "eps_switch_p25":    np.percentile(switches, 25),        "eps_switch_p75":    np.percentile(switches, 75),        "coverage":          len(switches)/MC_DRAWS,    })thresh_df = pd.DataFrame(threshold_rows)thresh_df.to_csv(OUTPUTS_DIR / "mc_thresholds.csv", index=False)print("Switching thresholds:")print(thresh_df.round(3).to_string(index=False))

## Section 8 — MILP Diagnostics

In [ ]:
# Summary stats on frontier qualityfeas_rate  = (frontier["status"]=="Optimal").mean()opt_rate   = feas_rate  # CBC certifies optimality on feasible instancessec_mean_col = "sec_mean"frontier[sec_mean_col] = frontier[SECONDARY_COLS].mean(axis=1)if len(feasible) > 0:    spread_primary = feasible["obj_value"].max() - feasible["obj_value"].min()    hv_proxy = (feasible["obj_value"] * feasible[sec_mean_col]).sum() / len(feasible)    nd_share = (feasible["obj_value"] == feasible["obj_value"].max()).mean()    print(f"Frontier diagnostics:")    print(f"  P(feasible) = {feas_rate:.3f}  |  Spread(primary) = {spread_primary:.3f}")    print(f"  Hypervolume proxy (2D) = {hv_proxy:.3f}")    print(f"  Non-dominated share = {nd_share:.3f}")

## Section 9 — Export Summary

In [ ]:
print("Files written to", OUTPUTS_DIR)for f in sorted(OUTPUTS_DIR.iterdir()):    size = f.stat().st_size    print(f"  {f.name:45s} {size:>8,} bytes")